# Credit Risk Early-Warning System: walkthrough

A two-stage credit risk framework for a small-ticket MSME / retail lender:

1. **Model 1, origination screening.** An ordinal logistic regression on application data (CIBIL, LTV, FOIR, income, ROI, product, profession, state, end use). It places each applicant on a 0–4 risk ladder.
2. **Model 2, behavioural early warning.** A logistic regression on EMI-bounce behaviour from April to June. It predicts which performing loans become NPAs (90+ days past due) by August.

> **Data note:** this notebook runs on **synthetic data** with the same structure as the original lender portfolio. No real borrower data is included, and the metrics differ from the production results.

In [1]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, "../src")
from credit_risk.synthetic import generate_portfolio
from credit_risk.features import standardise_bounce_reasons, behavioural_features, build_model_dataset, max_consecutive_streak
from credit_risk.models import ScreeningModel, MonitoringModel, risk_tier, score_borrower, score_to_band, BANDS
from credit_risk import evaluation as ev

loans, bounces = generate_portfolio(seed=42)
loans.shape, bounces.shape

((811, 15), (1026, 4))

## 1. Cleaning the bounce data
Bank and NACH files report the same failure in many spellings. These are mapped to five standard categories.

In [2]:
raw_counts = bounces["bounce_reason_raw"].str.strip().str.lower().value_counts()
print(raw_counts.head(10))
clean = standardise_bounce_reasons(bounces)
clean["bounce_reason"].value_counts()

bounce_reason_raw
not present                      178
not presented                    161
not to present                   160
insufficient funds                87
balance insufficient              84
4-balance insufficient            78
fund insuffieient                 75
insufficient balance              73
fund insufficient                 73
a/c blocked / frozen / closed      9
Name: count, dtype: int64


bounce_reason
not_presented      499
low_balance        470
account_issue       30
mandate_issue       20
technical_issue      7
Name: count, dtype: int64

## 2. Behavioural features
These are the number of months with a bounce, the longest run of consecutive bounce months, the number of low-balance bounces, and whether the loan bounced in the latest month.

**Observation window: April to June only.** The outcome is measured in August. Using July–August bounces would let the model see the default as it happens rather than predict it, which is target leakage.

In [3]:
print(max_consecutive_streak([4, 5, 6, 8]))   # -> 3
data = build_model_dataset(loans, bounces, observation_months=[4, 5, 6])
print(f"Loans modelled (performing in April): {len(data)},  new NPAs by August: {data['new_npa'].sum()}")
data[["loan_id", "bounce_frequency", "max_streak", "low_balance_bounces", "bounced_last_month", "new_npa"]].head()

3
Loans modelled (performing in April): 763,  new NPAs by August: 20


,loan_id,bounce_frequency,max_streak,low_balance_bounces,bounced_last_month,new_npa
0,SYN-00000,2,2,0,0,0
1,SYN-00001,0,0,0,0,0
2,SYN-00002,2,2,0,0,0
3,SYN-00003,2,2,1,0,0
4,SYN-00004,0,0,0,0,0


## 3. Model 1 target: an ordinal risk ladder
0 Healthy · 1 Low · 2 Moderate · 3 High (heavy bounces and worsening DPD) · 4 Severe (became an NPA)

In [4]:
full = behavioural_features(clean, [4, 5, 6, 7, 8]).reindex(data["loan_id"]).fillna(0).reset_index(drop=True)
tier = risk_tier(data, full["bounce_frequency"], full["max_streak"])
tier.value_counts().sort_index()

0    272
1    238
2    233
4     20
Name: count, dtype: int64

## 4. Out-of-sample validation (5-fold stratified cross-validation)
Every loan is scored by a model that never saw it in training.

In [5]:
from sklearn.model_selection import StratifiedKFold
y = data["new_npa"].to_numpy()
s1, s2 = np.zeros(len(data)), np.zeros(len(data))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=7).split(data, y):
    s1[te] = ScreeningModel().fit(data.iloc[tr], tier.iloc[tr]).score(data.iloc[te])
    s2[te] = MonitoringModel().fit(data.iloc[tr], data["new_npa"].iloc[tr]).score(data.iloc[te])

pd.DataFrame({"Model 1 - screening": ev.summary(y, s1), "Model 2 - early warning": ev.summary(y, s2)})

,Model 1 - screening,Model 2 - early warning
auc,0.724,0.833
ks,0.374,0.660
n_loans,763.000,763.000
n_defaults,20.000,20.000
capture_at_10pct_flagged,0.250,0.350
capture_at_20pct_flagged,0.450,0.850
capture_at_30pct_flagged,0.650,0.850


**How to read this:** `capture_at_20pct_flagged` is the share of loans that later defaulted that sit in the riskiest 20% of scores. This is the operational number for a collections team that can only call a fifth of the book.

In [6]:
ev.default_rate_by_band(score_to_band(s1), y, BANDS)

,loans,defaults,default_rate
band,,,
Very Low,35,0,0.000000
Low,217,2,0.009217
Moderate,257,5,0.019455
High,199,9,0.045226
Very High,55,4,0.072727


## 5. Final models and scoring a new borrower

In [7]:
screening = ScreeningModel().fit(data, tier)
monitoring = MonitoringModel().fit(data, data["new_npa"])
print(monitoring.coefficients())

score_borrower(
    screening, monitoring,
    applicant={"ltv": 0.60, "foir": 0.45, "tenure": 120, "roi": 13.5, "income": 90000, "mob": 18,
               "cibil": 720, "product": "HL", "profession": "Salaried", "state": "Delhi", "end_use": "Home Purchase"},
    behaviour={"bounce_frequency": 2, "max_streak": 2, "low_balance_bounces": 1, "bounced_last_month": 1},
)

bounce_frequency       0.367010
low_balance_bounces    0.579126
bounced_last_month     0.705319
max_streak             0.894883
dtype: float64


{'screening_score': 61.6,
 'screening_band': 'High',
 'early_warning_score': 83.1,
 'early_warning_band': 'Very High'}

## 6. Limitations
- **Small number of events.** Default counts are low (around 20 to 30), so the metrics have wide confidence intervals. A larger book or a longer history would be needed before production use.
- **Short horizon.** The model covers one 5-month window and has not been tested across economic cycles.
- **Fairness.** State and profession are used as predictors and should be reviewed for proxy discrimination before any automated decisioning. Model 1 is intended as a decision aid with a human reviewer, not an auto-reject.